# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review available record sets, fields, and their IDs.

Let's list all record sets defined in the Croissant metadata, and for each, print its `@id`, name, and available fields/columns.

In [ ]:
# Display all available record sets and their fields by @id
recordset_overview = []
if hasattr(metadata, 'record_sets'):
    for recset in metadata.record_sets:
        record_info = {
            "@id": getattr(recset, '@id', None),
            "name": getattr(recset, 'name', None),
            "fields": []
        }
        # Each recordset can have fields or columns
        if hasattr(recset, 'fields') and recset.fields:
            for fld in recset.fields:
                record_info["fields"].append({"@id": getattr(fld, "@id", None), "name": getattr(fld, "name", None)})
        # Croissant 1.x may use 'columns'
        if hasattr(recset, 'columns') and recset.columns:
            for col in recset.columns:
                record_info["fields"].append({"@id": getattr(col, "@id", None), "name": getattr(col, "name", None)})

        recordset_overview.append(record_info)

if len(recordset_overview) == 0:
    print("No record sets found in the metadata (property may be named '.recordSets' or '.record_sets').")
else:
    for r in recordset_overview:
        print(f"RecordSet @id: {r['@id']}" + (f", name: {r['name']}" if r['name'] else ""))
        if r['fields']:
            print("  Fields/Columns:")
            for f in r['fields']:
                print(f"    - @id: {f['@id']}", (f"name: {f['name']}" if f['name'] else ""))
        else:
            print("  No fields/columns found.")

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

**Note:** If no record sets are found, no data can be extracted (some minimal Croissant schemas may use a single default record set, or an implicit one).

In [ ]:
# Extract data from each record set (if record sets exist)
dataframes = {}
record_set_ids = [r['@id'] for r in recordset_overview if r['@id'] is not None]

if record_set_ids:
    for rec_id in record_set_ids:
        try:
            records = list(dataset.records(record_set=rec_id))
            df = pd.DataFrame(records)
            dataframes[rec_id] = df
            print(f"Loaded {len(df)} records from RecordSet @id: {rec_id}")
            print("Columns:", df.columns.tolist())
        except Exception as e:
            print(f"Error loading records for RecordSet @id: {rec_id}", e)
else:
    print("No record sets available to extract records.")

# Preview data for the first record set (if any)
if dataframes:
    first_rs_id = record_set_ids[0]
    print(f"\nSample data from RecordSet @id: {first_rs_id}")
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removal of outliers, transformation of distributions, or grouping data by key variables. Use field `@id` for column selection.

> _**Adjust the field `@id` below to numeric/categorical variables identified in the overview above. If no numeric fields are available, this cell will demonstrate structure only._

In [ ]:
# Pick a record set and a numeric field @id for demonstration
if dataframes:
    rec_id = list(dataframes.keys())[0]
    df = dataframes[rec_id].copy()

    # Try to guess a numeric column (commonly named '@id' like 'age', 'interval_months', etc):
    numeric_field_id = None
    for col in df.columns:
        if 'age' in col.lower() or 'interval' in col.lower() or ('months' in col.lower()) or (df[col].dtype in [int, float]):
            numeric_field_id = col
            break

    if numeric_field_id is None:
        print("No obvious numeric field detected. Review the record set field @ids above and set 'numeric_field_id' manually.")
    else:
        # Ensure numeric type
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].notna().any() else 0
        print(f"Filtering records where {numeric_field_id} > {threshold:.2f}")
        filtered_df = df[df[numeric_field_id] > threshold].copy()

        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalization (Z-score)
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std(ddof=0)
        print(f"Normalized {numeric_field_id} (Z-score):")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Group by another categorical field, if any
        group_field_id = None
        for col in df.columns:
            if col != numeric_field_id and df[col].dtype == object and df[col].nunique() < len(df) / 2:
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
else:
    print("No data available for EDA. Please ensure data extracting in previous step was successful.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset using field `@id` for columns. For example, plot histograms for numeric columns or bar plots for category frequencies.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    rec_id = list(dataframes.keys())[0]
    df = dataframes[rec_id].copy()
    # Pick numeric field and optional category
    numeric_field_id = None
    category_field_id = None
    for col in df.columns:
        if (df[col].dtype in [int, float]) or ('age' in col.lower()) or ('interval' in col.lower()):
            numeric_field_id = col
            break
    for col in df.columns:
        if col != numeric_field_id and df[col].nunique() > 1 and df[col].nunique() < len(df) / 2:
            category_field_id = col
            break

    # Plot histogram
    if numeric_field_id:
        plt.figure(figsize=(8, 4))
        sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
        plt.title(f"Distribution of {numeric_field_id} (@id)")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Frequency")
        plt.show()

    # Show barplot if a categorical field exits and less than 20 categories
    if category_field_id and numeric_field_id:
        plt.figure(figsize=(10, 4))
        sns.barplot(
            x=category_field_id, y=numeric_field_id,
            data=df.groupby(category_field_id)[numeric_field_id].mean().reset_index()
        )
        plt.title(f"Mean {numeric_field_id} by {category_field_id} (@id)")
        plt.xlabel(category_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion

In this notebook, we loaded a FAIR^2 Croissant dataset with `mlcroissant`, explored its record sets and fields via their `@id`, and demonstrated basic data extraction, transformation, and visualization workflows using pandas and seaborn.

- We referenced all entities by `@id` for clarity and reproducibility.
- Adjust the field and record set `@id` in the notebook for deeper analysis or more specific processing as desired.
- For complete documentation on the dataset, consult its source or embedded metadata for detailed schema and field descriptions.